In [1]:
#!/usr/bin/env python3

import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# USER SETTINGS
# ============================================================
NETID = os.environ.get("NETID", "k16v981")

DAILY_PEAK_GLOB = (
    f"/home/{NETID}/my_work/code/arabian_peninsula/"
    f"bayesian_extremes/data/DailyPeakState/DailyPeakState-*.nc"
)

PHASE_CSV = (
    f"/home/{NETID}/my_work/code/arabian_peninsula/"
    f"bayesian_extremes/data/sst/roni_dmi_monthly_1950_2025.csv"
)

OUT_DIR = Path(
    f"/home/{NETID}/my_work/code/arabian_peninsula/"
    f"bayesian_extremes/figures/phase_maps"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEMP_VAR = "t2m_at_wbt_daily_peak"
Q_VAR = "q_at_wbt_daily_peak"

LAT_MIN, LAT_MAX = 10, 34
LON_MIN, LON_MAX = 34, 60

PCTL = 0.95
MIN_COUNT = 10

USE_MONTHS = [6, 7, 8, 9]

# ------------------------------------------------------------
# Choose phase mode
# ------------------------------------------------------------
PHASE_MODE = "iod"   # "enso" or "iod"

# ENSO settings
ENSO_LAG = 2
ENSO_POS_THRESH = 0.5
ENSO_NEG_THRESH = -0.5

# IOD settings
IOD_LAG = 1
IOD_POS_THRESH = 0.5
IOD_NEG_THRESH = -0.5

# ============================================================
# HELPERS
# ============================================================
def classify_enso_from_roni(val):
    if pd.isna(val):
        return np.nan
    if val >= ENSO_POS_THRESH:
        return "El Nino"
    if val <= ENSO_NEG_THRESH:
        return "La Nina"
    return "Neutral"


def classify_iod_from_dmi(val):
    if pd.isna(val):
        return np.nan
    if val >= IOD_POS_THRESH:
        return "pIOD"
    if val <= IOD_NEG_THRESH:
        return "nIOD"
    return "Neutral"


def get_phase_config():
    mode = PHASE_MODE.lower()

    if mode == "enso":
        return {
            "mode": "enso",
            "raw_col": "RONI",
            "raw_name": "RONI_raw",
            "lag": ENSO_LAG,
            "lagged_col": "RONI_lagged",
            "phase_coord": "enso_phase_lagged",
            "classifier": classify_enso_from_roni,
            "pos_label": "El Nino",
            "neg_label": "La Nina",
            "pretty_name": "ENSO",
            "outfile_prefix": "enso",
        }

    if mode == "iod":
        return {
            "mode": "iod",
            "raw_col": "DMI",
            "raw_name": "DMI_raw",
            "lag": IOD_LAG,
            "lagged_col": "DMI_lagged",
            "phase_coord": "iod_phase_lagged",
            "classifier": classify_iod_from_dmi,
            "pos_label": "pIOD",
            "neg_label": "nIOD",
            "pretty_name": "IOD",
            "outfile_prefix": "iod",
        }

    raise ValueError("PHASE_MODE must be 'enso' or 'iod'")


def standardize_time_dim(ds: xr.Dataset) -> xr.Dataset:
    if "day" in ds.dims:
        ds = ds.rename({"day": "time"})
    if "day" in ds.coords and "time" not in ds.coords:
        ds = ds.rename({"day": "time"})

    ds = ds.sortby("latitude")
    ds = ds.sortby("longitude")

    ds = ds.sel(
        latitude=slice(LAT_MIN, LAT_MAX),
        longitude=slice(LON_MIN, LON_MAX)
    )
    return ds


def open_daily_peak_dataset() -> xr.Dataset:
    files = sorted(glob.glob(DAILY_PEAK_GLOB))
    if not files:
        raise FileNotFoundError(f"No files matched:\n{DAILY_PEAK_GLOB}")

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        preprocess=standardize_time_dim,
        engine="h5netcdf"
    )

    needed = ["wbt_daily_peak", TEMP_VAR, Q_VAR]
    missing = [v for v in needed if v not in ds.data_vars]
    if missing:
        raise ValueError(
            f"Missing required variables: {missing}\n"
            f"Available variables include: {list(ds.data_vars)}"
        )

    ds = ds.chunk({"time": -1})
    return ds


def load_phase_table(cfg):
    df = pd.read_csv(PHASE_CSV)

    if "time" not in df.columns:
        raise ValueError(
            f"Expected PHASE_CSV to have a 'time' column. "
            f"Available columns: {list(df.columns)}"
        )

    if cfg["raw_col"] not in df.columns:
        raise ValueError(
            f"Expected PHASE_CSV to have '{cfg['raw_col']}'. "
            f"Available columns: {list(df.columns)}"
        )

    df["time"] = pd.to_datetime(df["time"])
    df["ym"] = df["time"].dt.to_period("M")

    df = (
        df.sort_values("time")
          .groupby("ym", as_index=False)
          .first()
          .copy()
    )

    df["year"] = df["ym"].dt.year.astype(int)
    df["month"] = df["ym"].dt.month.astype(int)

    df = df.rename(columns={cfg["raw_col"]: cfg["raw_name"]})
    df = df.sort_values("ym").reset_index(drop=True)

    df[cfg["lagged_col"]] = df[cfg["raw_name"]].shift(cfg["lag"])
    df[cfg["phase_coord"]] = df[cfg["lagged_col"]].map(cfg["classifier"])

    if USE_MONTHS is not None:
        df = df[df["month"].isin(USE_MONTHS)].copy()

    return df


def attach_monthly_phases(ds: xr.Dataset, phase_df: pd.DataFrame, cfg) -> xr.Dataset:
    time_index = pd.to_datetime(ds["time"].values)
    ym = pd.Series(time_index).dt.to_period("M")

    if USE_MONTHS is not None:
        keep = pd.Series(time_index).dt.month.isin(USE_MONTHS).values
        ds = ds.isel(time=np.where(keep)[0])
        time_index = pd.to_datetime(ds["time"].values)
        ym = pd.Series(time_index).dt.to_period("M")

    phase_lookup = phase_df.set_index("ym")
    phase_vals = ym.map(phase_lookup[cfg["phase_coord"]]).to_numpy()

    ds = ds.assign_coords({
        cfg["phase_coord"]: ("time", phase_vals),
    })
    return ds


def phase_subset(ds: xr.Dataset, phase_coord: str, phase_label: str) -> xr.Dataset:
    mask = xr.DataArray(
        ds[phase_coord].values == phase_label,
        dims=("time",),
        coords={"time": ds["time"]}
    )
    out = ds.sel(time=mask)
    if out.sizes.get("time", 0) == 0:
        raise ValueError(f"No times found for {phase_coord} == {phase_label}")
    return out


def safe_quantile(da: xr.DataArray, q: float) -> xr.DataArray:
    if hasattr(da.data, "chunks") and da.chunks is not None:
        da = da.chunk({"time": -1})

    count = da.count("time")
    out = da.quantile(q, dim="time", skipna=True)
    out = out.where(count >= MIN_COUNT)

    if "quantile" in out.dims:
        out = out.squeeze("quantile", drop=True)

    return out


def compute_phase_p95_map(ds, phase_coord, phase_label, varname):
    sub = phase_subset(ds, phase_coord, phase_label)
    return safe_quantile(sub[varname], PCTL).rename(f"{varname}_{phase_label}_p95")


def compute_climatology_p95_map(ds, varname):
    return safe_quantile(ds[varname], PCTL).rename(f"{varname}_all_p95")


def nice_cbar_limit(*arrays, percentile=98):
    vals = []
    for arr in arrays:
        x = np.asarray(arr.values).ravel()
        x = x[np.isfinite(x)]
        if x.size:
            vals.append(x)

    if not vals:
        return 1.0

    vals = np.concatenate(vals)
    vmax = np.nanpercentile(np.abs(vals), percentile)
    return float(max(vmax, 0.25))


def add_map_features(ax):
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="0.92", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, linestyle="--", alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()


def plot_main_9panel(
    wbt_pos_anom, wbt_neg_anom, wbt_diff,
    t_pos_anom, t_neg_anom, t_diff,
    q_pos_anom, q_neg_anom, q_diff,
    cfg,
    outpath
):
    proj = ccrs.PlateCarree()

    fig, axes = plt.subplots(
        3, 3,
        figsize=(16, 12),
        subplot_kw={"projection": proj},
        constrained_layout=True
    )

    # convert q to g/kg
    q_pos_anom = (q_pos_anom * 1000.0).rename("q_pos_anom_gkg")
    q_neg_anom = (q_neg_anom * 1000.0).rename("q_neg_anom_gkg")
    q_diff     = (q_diff     * 1000.0).rename("q_diff_gkg")

    vmax_temp = nice_cbar_limit(
        wbt_pos_anom, wbt_neg_anom, wbt_diff,
        t_pos_anom, t_neg_anom, t_diff
    )
    norm_temp = TwoSlopeNorm(vmin=-vmax_temp, vcenter=0.0, vmax=vmax_temp)

    vmax_q = nice_cbar_limit(q_pos_anom, q_neg_anom, q_diff)
    norm_q = TwoSlopeNorm(vmin=-vmax_q, vcenter=0.0, vmax=vmax_q)

    fields = [
        (axes[0, 0], wbt_pos_anom, f"{cfg['pos_label']} anomaly\np95 WBT"),
        (axes[0, 1], wbt_neg_anom, f"{cfg['neg_label']} anomaly\np95 WBT"),
        (axes[0, 2], wbt_diff,     f"{cfg['neg_label']} − {cfg['pos_label']}\np95 WBT"),

        (axes[1, 0], t_pos_anom,   f"{cfg['pos_label']} anomaly\np95 t2m at WBT peak"),
        (axes[1, 1], t_neg_anom,   f"{cfg['neg_label']} anomaly\np95 t2m at WBT peak"),
        (axes[1, 2], t_diff,       f"{cfg['neg_label']} − {cfg['pos_label']}\np95 t2m at WBT peak"),

        (axes[2, 0], q_pos_anom,   f"{cfg['pos_label']} anomaly\np95 q at WBT peak"),
        (axes[2, 1], q_neg_anom,   f"{cfg['neg_label']} anomaly\np95 q at WBT peak"),
        (axes[2, 2], q_diff,       f"{cfg['neg_label']} − {cfg['pos_label']}\np95 q at WBT peak"),
    ]

    mappable_temp = None
    mappable_q = None

    for i, (ax, field, title) in enumerate(fields):
        add_map_features(ax)

        if i < 6:
            mappable_temp = ax.pcolormesh(
                field["longitude"], field["latitude"], field,
                transform=proj, cmap="coolwarm", norm=norm_temp, shading="auto"
            )
        else:
            mappable_q = ax.pcolormesh(
                field["longitude"], field["latitude"], field,
                transform=proj, cmap="BrBG", norm=norm_q, shading="auto"
            )

        ax.set_title(title, fontsize=11)

    cbar_temp = fig.colorbar(
        mappable_temp,
        ax=[axes[0, 0], axes[0, 1], axes[0, 2],
            axes[1, 0], axes[1, 1], axes[1, 2]],
        shrink=0.92,
        pad=0.03
    )
    cbar_temp.set_label("Difference from all-day p95 (°C)")

    cbar_q = fig.colorbar(
        mappable_q,
        ax=[axes[2, 0], axes[2, 1], axes[2, 2]],
        shrink=0.92,
        pad=0.03
    )
    cbar_q.set_label("Difference from all-day p95 (g/kg)")

    fig.suptitle(
        f"{cfg['pretty_name']} phase anomalies in p95 daily peak-state fields\n"
        f"{cfg['pretty_name']} lag = {cfg['lag']} month{'s' if cfg['lag'] != 1 else ''}",
        fontsize=14
    )

    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)


def main():
    cfg = get_phase_config()

    print("Opening DailyPeakState files...")
    ds = open_daily_peak_dataset()

    print(f"Loading and lagging monthly {cfg['pretty_name']} table...")
    phase_df = load_phase_table(cfg)

    print(f"Attaching monthly lagged {cfg['pretty_name']} phases to daily data...")
    ds = attach_monthly_phases(ds, phase_df, cfg)

    print("Computing p95 maps...")
    # phase-specific p95 maps
    wbt_pos = compute_phase_p95_map(ds, cfg["phase_coord"], cfg["pos_label"], "wbt_daily_peak")
    wbt_neg = compute_phase_p95_map(ds, cfg["phase_coord"], cfg["neg_label"], "wbt_daily_peak")

    t_pos = compute_phase_p95_map(ds, cfg["phase_coord"], cfg["pos_label"], TEMP_VAR)
    t_neg = compute_phase_p95_map(ds, cfg["phase_coord"], cfg["neg_label"], TEMP_VAR)

    q_pos = compute_phase_p95_map(ds, cfg["phase_coord"], cfg["pos_label"], Q_VAR)
    q_neg = compute_phase_p95_map(ds, cfg["phase_coord"], cfg["neg_label"], Q_VAR)

    # baseline p95 over all filtered days
    base_wbt = compute_climatology_p95_map(ds, "wbt_daily_peak")
    base_t   = compute_climatology_p95_map(ds, TEMP_VAR)
    base_q   = compute_climatology_p95_map(ds, Q_VAR)

    # anomalies relative to all-day p95
    wbt_pos_anom = (wbt_pos - base_wbt).rename(f"{cfg['outfile_prefix']}_wbt_pos_anom")
    wbt_neg_anom = (wbt_neg - base_wbt).rename(f"{cfg['outfile_prefix']}_wbt_neg_anom")
    wbt_diff     = (wbt_neg - wbt_pos).rename(f"{cfg['outfile_prefix']}_wbt_diff")

    t_pos_anom = (t_pos - base_t).rename(f"{cfg['outfile_prefix']}_t_pos_anom")
    t_neg_anom = (t_neg - base_t).rename(f"{cfg['outfile_prefix']}_t_neg_anom")
    t_diff     = (t_neg - t_pos).rename(f"{cfg['outfile_prefix']}_t_diff")

    q_pos_anom = (q_pos - base_q).rename(f"{cfg['outfile_prefix']}_q_pos_anom")
    q_neg_anom = (q_neg - base_q).rename(f"{cfg['outfile_prefix']}_q_neg_anom")
    q_diff     = (q_neg - q_pos).rename(f"{cfg['outfile_prefix']}_q_diff")

    print("Plotting main figure...")
    fig_path = OUT_DIR / f"{cfg['outfile_prefix']}_phase_p95_anoms_3col.png"
    plot_main_9panel(
        wbt_pos_anom, wbt_neg_anom, wbt_diff,
        t_pos_anom, t_neg_anom, t_diff,
        q_pos_anom, q_neg_anom, q_diff,
        cfg,
        fig_path
    )

    debug_ds = xr.Dataset({
        "base_wbt_p95": base_wbt,
        "base_t_p95": base_t,
        "base_q_p95": base_q,

        f"{cfg['outfile_prefix']}_wbt_pos_anom": wbt_pos_anom,
        f"{cfg['outfile_prefix']}_wbt_neg_anom": wbt_neg_anom,
        f"{cfg['outfile_prefix']}_wbt_diff": wbt_diff,

        f"{cfg['outfile_prefix']}_t_pos_anom": t_pos_anom,
        f"{cfg['outfile_prefix']}_t_neg_anom": t_neg_anom,
        f"{cfg['outfile_prefix']}_t_diff": t_diff,

        f"{cfg['outfile_prefix']}_q_pos_anom": q_pos_anom,
        f"{cfg['outfile_prefix']}_q_neg_anom": q_neg_anom,
        f"{cfg['outfile_prefix']}_q_diff": q_diff,
    })
    debug_ds.to_netcdf(OUT_DIR / f"{cfg['outfile_prefix']}_phase_map_products_3col.nc")

    print("\nDone.")
    print(f"Saved figure to: {fig_path}")
    print(f"Saved products to: {OUT_DIR}")


if __name__ == "__main__":
    main()

ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed


Opening DailyPeakState files...
Loading and lagging monthly IOD table...
Attaching monthly lagged IOD phases to daily data...
Computing p95 maps...
Plotting main figure...

Done.
Saved figure to: /home/k16v981/my_work/code/arabian_peninsula/bayesian_extremes/figures/phase_maps/iod_phase_p95_anoms_3col.png
Saved products to: /home/k16v981/my_work/code/arabian_peninsula/bayesian_extremes/figures/phase_maps
